# Auto Encoder
> https://excalidraw.com/#json=pWGS-CCIjK4BtIT32bs6F,T1KxjH7T1lPbDFhu8TnA7g

<img src="./images/thumbnail.png" alt="drawing" width="50%"/>

Given an input of $\text{input} \in \mathbb{R}^{D}$, one can represent the input data into smaller numbers or latents of size
$\text{latents} \in \mathbb{R}^{N}$ where $N \leq D$. Actually extremely simple with linear layers that progressively get smaller until we reach our 
desired latent, bottleneck size and then the reverse of linear layers that progressively project back to our original input size.

We can interpret the first part as basically "encoding" our data into $latent_size$ numbers that are then being decoded and 
reconstructed. 

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
print(device)

cuda


In [2]:
from PIL import Image
import numpy as np

# number 7, 8x8 image
inputsize = 8 * 8

img = Image.open("test.png").convert("L")
arr = np.array(img, dtype=np.float32) / 255.0  # Normalize to [0, 1]

tensor = torch.from_numpy(arr).reshape(1, inputsize)

The linear sizes are hard coded. All you need to know is that the input was an $8x8$ image that was converted into a
size 64 vector so the $\text{input} \in \mathbb{R}^{64}$. Not a big fan of ReLU due to dead neurons but it's simple and fast.

Sigmoid at the end to return numbers for $0\sim1$ as it is the same format as black white images. Not sure if LayerNorm actually helps but my reasoning was to scale the numbers such that the variance fits with the sigmoid function as to have better gradients. Maybe residuals are better but probably Conv2d is the true better choice.

In [3]:
# hyperparams
@dataclass
class autoencConfig:
    inputsize: int = 64
    bottleneck: int = 8
    lr: float = 3e-3

class AutoEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.encode = nn.Sequential(
                nn.Linear(config.inputsize, 32),
                nn.ReLU(),
                nn.Linear(32, 16),
                nn.ReLU(),
                nn.Linear(16, 8),
                nn.ReLU()
        )
        
        self.bottleneck = nn.Linear(8, config.bottleneck)
        
        self.decode = nn.Sequential(
            nn.ReLU(),
            nn.Linear(config.bottleneck, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, config.inputsize)
        )
        
        self.ln = nn.LayerNorm(config.inputsize)
        
        
    def forward(self, x, targets=None):
        x = self.encode(x)
        x = self.bottleneck(x)
        x = self.decode(x)
        
        if targets is None:
            loss = None
        else:
            loss = F.binary_cross_entropy_with_logits(x, targets)
        return x, loss

In [4]:
model = AutoEncoder(autoencConfig(inputsize=inputsize))
model.to(device=device)
optimizer = torch.optim.AdamW(model.parameters(), lr=model.config.lr)
outputs = []

In [ ]:
iters = 250

Xb = tensor.to(device=device)
for _ in range(iters):
    output, loss = model(Xb, Xb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    outputs.append(output)
    print(loss.item())
    


import matplotlib.pyplot as plt
from ipywidgets import interact

@interact(i=(0, len(outputs)-1))
def show_image(i):
    arr = outputs[i].cpu().detach().reshape(8, 8).numpy()
    # img = Image.fromarray((arr * 255).clip(0, 255).astype(np.uint8), mode="L")
    # img.save("reconstructed.png")
    
    plt.figure(figsize=(5,5))
    plt.imshow(arr, cmap="gray", vmin=0, vmax=1)
    plt.title(f"Output {i}")
    plt.show()
    # plt.axis("off")

0.6902360916137695
0.6839261054992676
0.6775091886520386
0.6708238124847412
0.6637582778930664
0.6563100814819336
0.6483404040336609
0.6396358013153076
0.6301210522651672
0.6195510029792786
0.6076862812042236
0.5942596793174744
0.5790127515792847
0.5615713596343994
0.5414464473724365
0.5181151628494263
0.49092167615890503
0.4598084092140198
0.424638032913208
0.38547593355178833
0.3427417278289795
0.29735901951789856
0.25084733963012695
0.2052895724773407
0.16310971975326538
0.1266230195760727
0.09746469557285309
0.076121486723423
0.06174559146165848
0.05299681797623634
0.04813753813505173
0.04560849443078041
0.0443878099322319
0.04380546137690544
0.04354894906282425
0.04342888295650482
0.04338669776916504
0.04334928095340729
0.04335571452975273
0.043338701128959656
0.04334568977355957
0.04333528131246567
0.043343398720026016
0.04333805292844772
0.043340060859918594
0.04333003982901573
0.043343301862478256
0.043330710381269455
0.04333154857158661
0.043339721858501434
0.0433240607380867


interactive(children=(IntSlider(value=124, description='i', max=249), Output()), _dom_classes=('widget-interac…